In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset

In [2]:
VOCAB_SIZE = 12000
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 13
BATCH_SIZE = 64
EPOCHS = 5
MAX_LEN = 60

In [4]:
print("Loading dair-ai/emotion....")
data=load_dataset("dair-ai/emotion")

Loading dair-ai/emotion....


In [7]:
X_train_raw = list(data["train"]["text"])
y_train_strings = list(data["train"]["label"])

In [8]:
label_to_id = {
    "sadness": 0,
    "happiness": 1,
    "love": 2,
    "anger": 3,
    "fear": 4,
    "surprise": 5,
    "shame": 6,
    "guilt": 7,
    "disgust": 8,
    "confusion": 9,
    "boredom": 10,
    "relief": 11,
    "sarcasm": 12
}

In [9]:
y_all_labels = np.array([
    label_to_id.get(
        str(label).lower().strip(),
        0
    )
    for label in y_train_strings
], dtype=np.int32)

In [15]:
if "validation" in data:
    X_train_raw = list(data["train"]["text"])
    y_train_strings = list(data["train"]["label"])
    X_val_raw = list(data["validation"]["text"])
    y_val_strings = list(data["validation"]["label"])
    y_val = np.array([
        label_to_id.get(
            str(label).lower().strip(),
            0
        )
        for label in y_val_strings
    ], dtype=np.int32)
    y_train = y_all_labels  
else:
    total_len = len(X_train_raw)
    val_size = int(total_len * 0.1)
    split_idx = total_len - val_size
    X_val_raw = X_train_raw[split_idx:]
    X_train_raw = X_train_raw[:split_idx]
    y_val = y_all_labels[split_idx:]
    y_train = y_all_labels[:split_idx]
print(f"Training samples: {len(X_train_raw)}")
print(f"Validation samples: {len(X_val_raw)}")


Training samples: 16000
Validation samples: 2000


In [16]:
print("\nTokenizing text...")
VOCAB_SIZE = 12000
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)
tokenizer.fit_on_texts(X_train_raw)
X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
X_val_seq = tokenizer.texts_to_sequences(X_val_raw)
print("Tokenization completed.")
print("First training sequence:", X_train_seq[0])
print("First validation sequence:", X_val_seq[0])


Tokenizing text...
Tokenization completed.
First training sequence: [2, 139, 3, 679]
First validation sequence: [17, 8, 157, 260, 4, 343, 16, 51, 19, 212, 11289, 50, 10, 13, 533]


In [17]:
print("Padding sequences...")
X_train = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)
X_val = pad_sequences(
    X_val_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)
print("Preprocessing complete.")
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

Padding sequences...
Preprocessing complete.
X_train shape: (16000, 60)
X_val shape: (2000, 60)


In [18]:
print("\nBuilding LSTM model...")
model = Sequential([
        tf.keras.layers.Input(
        shape=(MAX_LEN,)
    ),
       Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
 LSTM(
        units=HIDDEN_DIM,
        return_sequences=False
    ),
    Dropout(0.3),
Dense(
        units=OUTPUT_DIM,
        activation="softmax"
    )
])


Building LSTM model...


In [19]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 60, 100)             │       1,200,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 128)                 │         117,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 13)                  │           1,677 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,318,925 (5.03 MB)

 Trainable params: 1,318,925 (5.03 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
print("\nStarting training...")
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)


Starting training...
Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 27s 95ms/step - accuracy: 0.9977 - loss: 0.0426 - val_accuracy: 1.0000 - val_loss: 1.8639e-05
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 23s 93ms/step - accuracy: 1.0000 - loss: 2.9619e-05 - val_accuracy: 1.0000 - val_loss: 1.0273e-05
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 24s 94ms/step - accuracy: 1.0000 - loss: 1.7408e-05 - val_accuracy: 1.0000 - val_loss: 6.3309e-06
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 23s 94ms/step - accuracy: 1.0000 - loss: 1.1177e-05 - val_accuracy: 1.0000 - val_loss: 4.0616e-06
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 24s 95ms/step - accuracy: 1.0000 - loss: 8.1431e-06 - val_accuracy: 1.0000 - val_loss: 3.1051e-06


In [29]:
import numpy as np

label_mapping = {
    0: "sadness",
    1: "happiness",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise",
    6: "shame",
    7: "guilt",
    8: "disgust",
    9: "confusion",
    10: "boredom",
    11: "relief",
    12: "sarcasm"
}

In [22]:
def predict_emotion(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )
    prediction = model.predict(
        padded_sequence,
        verbose=0
    )
    predicted_class = np.argmax(prediction[0])
    emotion = label_mapping[predicted_class]
    confidence = prediction[0][predicted_class]
    return emotion, confidence

In [23]:
def get_sentiment(emotion):
    if emotion in [
        "happiness",
        "love",
        "relief"
    ]:
        return "Positive"
    elif emotion in [
        "sadness",
        "anger",
        "fear",
        "shame",
        "guilt",
        "disgust",
        "boredom"
    ]:
        return "Negative"
    else:
        return "Neutral/Complex"

In [39]:
def predict_emotion(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )
    prediction = model.predict(padded_sequence, verbose=0)
    predicted_class = np.argmax(prediction[0])
    emotion = label_mapping[predicted_class]
    confidence = prediction[0][predicted_class]
    return emotion, confidence

In [40]:
def get_sentiment(emotion):
    if emotion in ["happiness", "love", "relief"]:
        return "Positive"
    elif emotion in ["sadness", "anger", "fear", "shame", "guilt", "disgust", "boredom"]:
        return "Negative"
    else:
        return "Neutral/Complex"

In [41]:
def predict_emotion_and_sentiment(text):
    emotion, confidence = predict_emotion(text)
    sentiment = get_sentiment(emotion)
    return emotion, sentiment, confidence

In [42]:
text = "I am sad today."
emotion, sentiment, confidence = predict_emotion_and_sentiment(text)
print("\n================================")
print("       MODEL PREDICTION")
print("================================")
print("Input Text:", text)
print("Predicted Emotion:", emotion)
print("Sentiment:", sentiment)
print("Confidence:", round(float(confidence) * 100, 2), "%")


       MODEL PREDICTION
Input Text: I am sad today.
Predicted Emotion: sadness
Sentiment: Negative
Confidence: 100.0 %
